## Regional SAT anomalies calculation then calculate the trend

In [ ]:
# In[1]:
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
# %%
# define function
import src.SAT_function_Obs_Fingerprint as data_process
import src.Data_Preprocess as preprocess

In [ ]:
# import src.slurm_cluster as scluster
# client, scluster = scluster.init_dask_slurm_cluster(scale=2,cores=10, memory="200GB")

In [ ]:
def func_mk(x):
    """
    Mann-Kendall test for trend
    """
    results = data_process.mk_test(x)
    slope = results[0]
    p_val = results[1]
    return slope, p_val

In [ ]:
# load data
OBS_data = ['Berkeley', 'NOAA']
OBS = OBS_data[1]
dir_observation = '/work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG2/'
observation = xr.open_dataset(dir_observation + f'Raw_{OBS}_MK_trend_1950-2022_sliding.nc')

# dir_forced_unforced_anom = '/work/mh0033/m301036/OBS_LPS_revision/docs/data/FIGS5_S6/cesm2_100/'
# observation_partition_anom = xr.open_mfdataset(dir_forced_unforced_anom + 'OBS_SAT_anomaly_partition_wrt_MMLE_ENS.nc',chunks={'lat':10,'lon':10})
dir_forced_unforced = '/work/mh0033/m301036/OBS_LPS_revision/docs/data/FIGS_OBS_records/'
observation_forced = xr.open_mfdataset(dir_forced_unforced + f'/trend_forced_{OBS}_annual/' + f'forced_{OBS}_MMLE_MK_trend_1950-2022_sliding.nc',chunks={'lat':10,'lon':10}) 
observation_internal = xr.open_mfdataset(dir_forced_unforced + f'/trend_ICV_{OBS}_annual/' + f'ICV_{OBS}_MMLE_MK_trend_1950-2022_sliding.nc',chunks={'lat':10,'lon':10})

In [ ]:
observation['trend'].sel(period='2013-2022').plot()

In [ ]:
observation_forced, observation_internal 

In [ ]:
observation_forced['trend'].sel(period='2013-2022').plot()

In [ ]:
observation_internal['trend'].sel(period='2013-2022').plot()

### Calculate the trend end year fix to 2022, start with 73 year length and decrease length of trend every one year, the minimum trend length is 10yr 

In [ ]:
temp_data = observation.trend
temp_data_forced = observation_forced.trend
temp_data_internal = observation_internal.trend

### Regional anomalies calculation

In [ ]:
def plot_trend(temp_data, lats, lons, levels=None, extend=None, cmap=None, 
                                 title="", ax=None, show_xticks=False, show_yticks=False):
    """
    Plot the trend spatial pattern using Robinson projection with significance overlaid.

    Parameters:
    - temp_data: 2D numpy array with the trend values.
    - lats, lons: 1D arrays of latitudes and longitudes.
    - p_values: 2D array with p-values for each grid point.
    - GMST_p_values: 2D array with GMST p-values for each grid point.
    - title: Title for the plot.
    - ax: Existing axis to plot on. If None, a new axis will be created.
    - show_xticks, show_yticks: Boolean flags to show x and y axis ticks.
    
    Returns:
    - contour_obj: The contour object from the plot.
    """
    # Plotting
    contour_obj = ax.contourf(lons, lats, temp_data, levels=levels, extend=extend, cmap=cmap, transform=ccrs.PlateCarree())

    ax.coastlines(resolution='110m')
    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False,
                      color='gray', alpha=0.35, linestyle='--')

    # Disable labels on the top and right of the plot
    gl.top_labels = False
    gl.right_labels = False

    # Enable labels on the bottom and left of the plot
    gl.bottom_labels = show_xticks
    gl.left_labels = show_yticks
    gl.xformatter = cticker.LongitudeFormatter()
    gl.yformatter = cticker.LatitudeFormatter()
    gl.xlabel_style = {'size': 16}
    gl.ylabel_style = {'size': 16}
    
    if show_xticks:
        gl.bottom_labels = True
    if show_yticks:
        gl.left_labels = True
    
    # ax.set_title(title, loc='center', fontsize=18, pad=5.0)

    return contour_obj
# %%
plt.rcParams['figure.figsize'] = (8, 10)
plt.rcParams['font.size'] = 16
# plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.labelsize'] = 16
plt.rcParams['ytick.direction'] = 'out'
plt.rcParams['ytick.minor.visible'] = True
plt.rcParams['ytick.major.right'] = True
plt.rcParams['ytick.right'] = True
plt.rcParams['xtick.bottom'] = True
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['savefig.bbox'] = 'tight'
plt.rcParams['savefig.pad_inches'] = 0.1
plt.rcParams['savefig.transparent'] = True

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.ticker as mticker
import cartopy.feature as cfeature
import cartopy.mpl.ticker as cticker
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import matplotlib.gridspec as gridspec
import matplotlib as mpl
import seaborn as sns
from matplotlib.colors import ListedColormap
from matplotlib.colors import BoundaryNorm, ListedColormap
import cartopy.util as cutil
import seaborn as sns
import matplotlib.colors as mcolors
import palettable

In [ ]:
lat = temp_data.lat
lon = temp_data.lon

In [ ]:
# North Pacific sector region
lat1 = 30
lat2 = 50
lon1 = 175  #180
lon2 = 220  #260

In [ ]:
temp_data_NPI,lons_NPI, lats_NPI = data_process.selreg(
        temp_data, lat, lon, lat1=lat1, lat2=lat2, lon1=lon1, 
        lon2=lon2)
temp_forced_da_NPI,lons_NPI, lats_NPI = data_process.selreg(
        temp_data_forced, lat, lon, lat1=lat1, lat2=lat2, lon1=lon1, 
        lon2=lon2)

temp_internal_da_NPI,lons_NPI, lats_NPI = data_process.selreg(
        temp_data_internal, lat, lon, lat1=lat1, lat2=lat2, lon1=lon1, 
        lon2=lon2)  

In [ ]:
temp_forced_da_NPI_mean = data_process.calc_weighted_mean(temp_forced_da_NPI)
temp_internal_da_NPI_mean = data_process.calc_weighted_mean(temp_internal_da_NPI)

In [ ]:
# calculate the NPI region SAT anomalies
temp_da_NPI_mean = data_process.calc_weighted_mean(temp_data_NPI)

In [ ]:
temp_da_NPI_mean

In [ ]:
temp_forced_da_NPI_mean

In [ ]:
import os
dir_out = f'/work/mh0033/m301036/OBS_LPS_revision/docs/data/FIGS_OBS_records/regional_data/{OBS}/'
os.makedirs(dir_out, exist_ok=True)

temp_da_NPI_mean.to_dataset(name='trend').to_netcdf(dir_out + f'Raw_{OBS}_NPI_trend_1950-2022_sliding.nc')
temp_forced_da_NPI_mean.to_dataset(name='trend').to_netcdf(dir_out + f'Forced_{OBS}_NPI_trend_1950-2022_sliding.nc')
temp_internal_da_NPI_mean.to_dataset(name='trend').to_netcdf(dir_out + f'Internal_{OBS}_NPI_trend_1950-2022_sliding.nc')

### Segments definitions

In [ ]:
# calculate the trend for each segment
trends = {}
for begin_year in temp_da_NPI_mean.period.values:
    trends[begin_year] = {
        "raw": temp_da_NPI_mean.sel(period=begin_year).values,
        "forced": temp_forced_da_NPI_mean.sel(period=begin_year).values,
        "internal": temp_internal_da_NPI_mean.sel(period=begin_year).values
    }

In [ ]:
trends_df = pd.DataFrame.from_dict(trends, orient='index')
trends_df

In [ ]:
# data frame to dataset
trends_ds = trends_df.to_xarray()

In [ ]:
trends_ds

In [ ]:
# export the trend to the netcdf file
import os
dir_out = f'/work/mh0033/m301036/OBS_LPS_revision/docs/data/FIGS_OBS_records/regional_data/{OBS}/'
os.makedirs(dir_out, exist_ok=True)
trends_ds.to_netcdf(dir_out + f'NPI_{OBS}_trend_variations.nc')

In [ ]:
dirin = f'/work/mh0033/m301036/OBS_LPS_revision/docs/data/FIGS_OBS_records/regional_data/{OBS}/percentile/'
NPI_unforced_lower = xr.open_dataset(f'{dirin}internal_NPI_trend_lower_percentile.nc')
NPI_unforced_upper = xr.open_dataset(f'{dirin}internal_NPI_trend_upper_percentile.nc')

In [ ]:
NPI_unforced_lower

In [ ]:
import seaborn as sns
from matplotlib.lines import Line2D

# sns.set_theme(style="whitegrid")
# Set the font dictionaries (for plot title and axis titles)
title_font = {'fontname': 'Arial', 'size': '20', 'color': 'black', 'weight': 'normal',
                'verticalalignment': 'bottom'}  # Bottom vertical alignment for more space
axis_font = {'fontname': 'Arial', 'size': '20'}

# Create the plot
fig = plt.figure(figsize=(25, 15))
gs = gridspec.GridSpec(2, 2, wspace=0.25, hspace=0.7)

ax1 = plt.subplot(gs[0, 0])
# ax2 = plt.subplot(gs[0, 1])
# ax3 = plt.subplot(gs[1, 0])
# ax4 = plt.subplot(gs[1, 1])

# define rgb colors for the outlines
# colors = [(32,120,180), #blue
#           (106,61,154), #purple
#           (173,23,88), #magenta
#           (255,127,0), #orange
#           (226,26,27),#red
#           (49,160,45) #green
#          ]
# colors_set = [(r / 255, g / 255, b / 255) for r, g, b in colors]
colors = ['#0F1023','#B11927', '#407BD0', '#B7D0EA']
line_widths = [5.5, 5.5, 5.5, 5.5, 1.5, 1.5, 1.5]
titles = ['North Pacific sector(NPI)']
linestyles = ['-', '-', '-.', ':']

years = np.arange(1950, 2014)

vars = ['raw', 'forced', 'internal']
# We use a loop to simulate multiple lines for each category
for i, var in enumerate(vars):
    y_vals = trends_ds[var].astype(float).values
    sns.lineplot(x=years, y=y_vals, color=colors[i], linestyle=linestyles[i], linewidth=3.5, ax=ax1)

ax1.fill_between(years, NPI_unforced_lower.ICV_trend_lower.values[::-1], NPI_unforced_upper.ICV_trend_upper.values[::-1], color=colors[3])

ax1.set_xlim([1945, 2015])
ax1.set_ylim([-1.0, 1.0])
ax1.set_xticks([1953, 1963, 1973, 1983, 1993, 2003, 2013])
ax1.set_xticklabels(['1953', '1963', '1973', '1983', '1993', '2003', '2013'])
ax1_upper = ax1.twiny()
ax1_upper.invert_xaxis()
ax1_upper.set_xlim([78,8])
ax1_upper.set_xlabel('Length of trends', fontsize=28, labelpad=10)
ax1_upper.set_xticks([70, 60, 50, 40, 30, 20, 10])
ax1_upper.set_xticklabels(['70', '60', '50', '40', '30', '20', '10'])
ax1.spines['top'].set_linewidth(2.5)
ax1.spines['right'].set_linewidth(2.5)
ax1.spines['bottom'].set_linewidth(2.5)
ax1.spines['left'].set_linewidth(2.5)
ax1_upper.tick_params(axis='x', labelsize=26)
ax1_upper.tick_params(axis='x', which='major', length=12, width=2.5, direction='in')
ax1.axhline(y=0, color='grey', linestyle='--', linewidth=2.5, alpha=0.75)
ax1.set_ylabel('Trend value (°C/decade)', fontsize=30)
ax1.set_xlabel('Start year of linear trend', fontsize=30)
ax1.set_title(titles[0], loc='left',fontsize=32,pad=20)
ax1.tick_params(axis='x', which='major', length=12, labelsize=26, width=2.5, direction='in')
ax1.tick_params(axis='y', which='major', length=12, labelsize=26, width=2.5, direction='in')
ax1.axvline(x=2013, color='#999A9E', linestyle='-', linewidth=2.5, alpha=0.75)
ax1.axvline(x=1993, color='#999A9E', linestyle='-', linewidth=2.5, alpha=0.75)
ax1.axvline(x=1979, color='#999A9E', linestyle='-', linewidth=2.5, alpha=0.75)
ax1.axvline(x=1963, color='#999A9E', linestyle='-', linewidth=2.5, alpha=0.75)
ax1.text(1979.1, -0.96, '1979-2022', fontsize=26, rotation=90, color='#999A9E')
# ax1.text(1940, 1.58, 'c', fontsize=42, ha='center', va='center', fontweight='bold')
# custom_lines = [Line2D([0], [0], color=colors[0], lw=3.5),
#                 Line2D([0], [0], color=colors[1], lw=3.5),
#                 Line2D([0], [0], color=colors[2], lw=3.5)]
# leg2 = ax1.legend(custom_lines, ['total', 'human forced', 'internal variability'], 
#                   loc='lower left', fontsize=26)
# ax1.add_artist(leg2)
plt.savefig('./NPI_trend_variations.png', dpi=300, bbox_inches='tight')
plt.show()